# NBGrader - Sistema Automatizado de Calificación

## Sistema Dinámico y Configurable

Este notebook permite configurar cualquier curso, assignment y lista de estudiantes de forma dinámica.

## Observaciones Importantes:
1. Se debe reiniciar la sesión si se copian archivos nuevos de estudiantes
2. Los nombres de carpetas NO pueden tener acentos (el feedback no funciona)
3. Todos los parámetros son configurables al inicio

## Workflow:
1. Configurar parámetros del curso
2. Cargar estudiantes desde CSV
3. Generar assignment
4. Calificar automáticamente
5. Generar feedback y exportar notas

## 1. Instalación de Dependencias

In [ ]:
# Instalar nbclient 0.6.1
!pip install nbclient==0.6.1 -q

In [ ]:
# Instalar nbgrader 0.8.1
!pip install nbgrader==0.8.1 -q

## 2. Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Configuración Dinámica del Curso

**IMPORTANTE:** Configura aquí todos los parámetros de tu curso

In [ ]:
import os
import pandas as pd
import shutil
import re
from pathlib import Path

# ============================================================================
# CONFIGURACIÓN DINÁMICA - PERSONALIZA ESTOS VALORES
# ============================================================================

# 1. RUTA BASE EN GOOGLE DRIVE
# Esta es la carpeta principal donde se organizarán TODOS tus cursos
# Ejemplos: '/content/drive/MyDrive/nbgrader', '/content/drive/MyDrive/Cursos'
print("💡 La ruta base es la carpeta principal para TODOS tus cursos")
print("   Dentro se creará una subcarpeta para cada curso")
print("   Ejemplo: /content/drive/MyDrive/nbgrader\n")
BASE_PATH_INPUT = input("Ingresa la ruta base en Google Drive: ").strip()
if not BASE_PATH_INPUT:
    BASE_PATH_INPUT = '/content/drive/MyDrive/nbgrader'
    print(f"⚠️  Usando valor por defecto: {BASE_PATH_INPUT}")

# 2. NOMBRE DEL CURSO (se crea como carpeta dentro de BASE_PATH)
# Ejemplos: 'Python_AP', 'DataScience_101', 'MachineLearning_2024'
print("\n💡 El nombre del curso se creará como carpeta dentro de la ruta base")
print("   Ejemplo: Si usas 'Python_2024', se creará:")
print(f"   {BASE_PATH_INPUT}/Python_2024/\n")
COURSE_ID = input("Ingresa el ID del curso (ej: Python_AP): ").strip()
if not COURSE_ID:
    COURSE_ID = 'MiCurso'
    print(f"⚠️  Usando valor por defecto: {COURSE_ID}")

# 3. ID DEL ASSIGNMENT
# Ejemplos: 'S01_D02_A02', 'Tarea1', 'Parcial_Final'
ASSIGNMENT_ID = input("\nIngresa el ID del assignment (ej: S01_D02_A02): ").strip()
if not ASSIGNMENT_ID:
    ASSIGNMENT_ID = 'Assignment1'
    print(f"⚠️  Usando valor por defecto: {ASSIGNMENT_ID}")

# 4. TIMEOUT DE EJECUCIÓN (en segundos)
# Recomendado: 180 (3 minutos) - Aumentar si los ejercicios son complejos
TIMEOUT = input("\nTimeout de ejecución en segundos [180]: ").strip()
TIMEOUT = int(TIMEOUT) if TIMEOUT else 180

# Validar que no haya espacios ni caracteres especiales
if ' ' in COURSE_ID or ' ' in ASSIGNMENT_ID:
    print("\n❌ ERROR: Los IDs no pueden contener espacios")
    print("   Usa guiones bajos en su lugar: Mi_Curso en vez de 'Mi Curso'")
    raise ValueError("IDs con espacios no permitidos")

# CONSTRUIR RUTA COMPLETA: BASE_PATH/COURSE_ID
BASE_PATH = os.path.join(BASE_PATH_INPUT, COURSE_ID)

# Rutas de directorios (dentro de BASE_PATH/COURSE_ID/)
DIRS = {
    'source': os.path.join(BASE_PATH, 'source'),
    'release': os.path.join(BASE_PATH, 'release'),
    'submitted': os.path.join(BASE_PATH, 'submitted'),
    'autograded': os.path.join(BASE_PATH, 'autograded'),
    'feedback': os.path.join(BASE_PATH, 'feedback')
}

print("\n" + "="*70)
print("CONFIGURACIÓN DEL CURSO")
print("="*70)
print(f"📚 Curso:       {COURSE_ID}")
print(f"📝 Assignment:  {ASSIGNMENT_ID}")
print(f"📂 Ruta base:   {BASE_PATH_INPUT}")
print(f"📁 Ruta curso:  {BASE_PATH}")
print(f"⏱️  Timeout:     {TIMEOUT} segundos")
print("="*70)
print(f"\n💡 Estructura que se creará:")
print(f"   {BASE_PATH}/")
print(f"   ├── source/")
print(f"   ├── release/")
print(f"   ├── submitted/")
print(f"   ├── autograded/")
print(f"   └── feedback/")
print("="*70)
print("\n✅ Configuración lista!")

## 4. Generar Archivo de Configuración de NBGrader

Esta celda crea automáticamente el archivo `nbgrader_config.py` con tu configuración.

In [ ]:
# Generar nbgrader_config.py dinámicamente
config_content = f'''# ============================================================================
# NBGrader Configuration File - Generado Automáticamente
# ============================================================================
# Curso: {COURSE_ID}
# Assignment: {ASSIGNMENT_ID}
# ============================================================================

# Directorio raíz del curso
c.CourseDirectory.root = '{BASE_PATH}'

# Configuración de ejecución
c.Execute.timeout = {TIMEOUT}
c.Execute.ipython_hist_file = ':memory:'
c.Execute.record_timing = True

# Configuración de soluciones y tests
c.ClearSolutions.begin_solution_delimeter = 'BEGIN SOLUTION'
c.ClearSolutions.end_solution_delimeter = 'END SOLUTION'
c.ClearSolutions.code_stub = {{
    'python': '# YOUR CODE HERE\\nraise NotImplementedError()',
    'r': '# YOUR CODE HERE\\nstop("No Answer Given!")'
}}

# Tests ocultos
c.ClearHiddenTests.begin_test_delimeter = 'BEGIN HIDDEN TESTS'
c.ClearHiddenTests.end_test_delimeter = 'END HIDDEN TESTS'
c.ClearHiddenTests.enforce_metadata = False

# Bloqueo de celdas
c.LockCells.lock_grade_cells = True
c.LockCells.lock_readonly_cells = True
c.LockCells.lock_solution_cells = True

# Límites de output
c.LimitOutput.max_lines = 1000
c.LimitOutput.max_traceback = 100

# Archivos a ignorar
c.CourseDirectory.ignore = [
    '.ipynb_checkpoints',
    '*.pyc',
    '__pycache__',
    'feedback',
    '.DS_Store'
]

# Tamaño máximo de archivos (100 MB)
c.CourseDirectory.max_file_size = 100000

# Penalizaciones por entrega tardía
c.LateSubmissionPlugin.penalty_method = 'none'

# Timestamps
c.Exchange.timestamp_format = '%Y-%m-%d %H:%M:%S %Z'
c.Exchange.timezone = 'UTC'

# Feedback
c.GetGrades.display_data_priority = [
    'text/html',
    'application/pdf',
    'text/latex',
    'image/svg+xml',
    'image/png',
    'image/jpeg',
    'text/plain'
]
'''

# Escribir archivo
config_path = '/content/nbgrader_config.py'
with open(config_path, 'w') as f:
    f.write(config_content)

print(f"✅ Archivo de configuración generado: {config_path}")
print(f"\n📋 Contenido:")
print("="*70)
!head -20 /content/nbgrader_config.py
print("...")
print("="*70)

## 5. Crear Estructura de Directorios

In [ ]:
# Crear directorios base del curso
print("Creando estructura de directorios...\n")
for dir_name, dir_path in DIRS.items():
    try:
        os.makedirs(dir_path, exist_ok=True)
        print(f"✓ {dir_name:12} → {dir_path}")
    except OSError as error:
        print(f"✗ {dir_name:12} → Error: {error}")

print(f"\n✅ Estructura creada en: {BASE_PATH}")

## 6. Cargar Lista de Estudiantes desde CSV

### Formato del CSV esperado:
```csv
nombre_estudiante
Apellido1_Apellido2_Nombre1_Nombre2
...
```

**IMPORTANTE:** Los nombres NO deben tener acentos ni caracteres especiales.

In [ ]:
from google.colab import files

# Subir archivo CSV
print("📤 Por favor, sube el archivo CSV con la lista de estudiantes")
print("   Formato: nombre_estudiante (una columna, sin acentos)\n")
uploaded = files.upload()

# Leer CSV
csv_filename = list(uploaded.keys())[0]
df_estudiantes = pd.read_csv(csv_filename)

# Validar que existe la columna requerida
if 'nombre_estudiante' not in df_estudiantes.columns:
    raise ValueError("❌ El CSV debe tener una columna llamada 'nombre_estudiante'")

# Obtener lista de estudiantes
estudiantes = df_estudiantes['nombre_estudiante'].tolist()

# Validar nombres (sin acentos ni espacios)
print("\n🔍 Validando nombres de estudiantes...\n")
errores_validacion = []

for est in estudiantes:
    # Convertir a string por si acaso
    est_str = str(est).strip()
    
    # Verificar espacios
    if ' ' in est_str:
        errores_validacion.append(f"   ⚠️  '{est_str}' contiene espacios (usar guiones bajos)")
    
    # Verificar caracteres especiales o acentos
    if not re.match(r'^[a-zA-Z0-9_]+$', est_str):
        errores_validacion.append(f"   ⚠️  '{est_str}' contiene acentos o caracteres especiales")

if errores_validacion:
    print("❌ ERRORES DE VALIDACIÓN:")
    for error in errores_validacion:
        print(error)
    print("\n⚠️  Los nombres deben:")
    print("   - Usar guiones bajos en lugar de espacios")
    print("   - NO tener acentos (ej: Martinez en vez de Martínez)")
    print("   - Solo letras, números y guiones bajos")
    raise ValueError("CSV con nombres inválidos")

print("="*70)
print(f"✅ Se cargaron {len(estudiantes)} estudiantes válidos")
print("="*70)
for i, est in enumerate(estudiantes, 1):
    print(f"  {i:2}. {est}")
print("="*70)

### Opción alternativa: Definir estudiantes manualmente

In [ ]:
# Descomentar si prefieres definir la lista manualmente
# estudiantes = [
#     'Estudiante_Uno',
#     'Estudiante_Dos',
#     'Estudiante_Tres'
# ]
# print(f"✓ Lista manual: {len(estudiantes)} estudiantes")

## 7. Crear Carpetas de Estudiantes

In [ ]:
# Crear carpetas para cada estudiante en 'submitted'
submitted_base = DIRS['submitted']

print(f"Creando carpetas para {len(estudiantes)} estudiantes...\n")

for estudiante in estudiantes:
    # Crear carpeta del estudiante
    student_path = os.path.join(submitted_base, estudiante)
    os.makedirs(student_path, exist_ok=True)
    
    # Crear subcarpeta del assignment
    assignment_path = os.path.join(student_path, ASSIGNMENT_ID)
    os.makedirs(assignment_path, exist_ok=True)
    
    print(f"✓ {estudiante}/{ASSIGNMENT_ID}")

print(f"\n✅ Carpetas creadas en: {submitted_base}")

## 8. Generar Assignment para Estudiantes

**Prerequisito:** Debes tener el notebook fuente en:
```
source/{ASSIGNMENT_ID}/{ASSIGNMENT_ID}.ipynb
```

In [ ]:
# Verificar que existe el archivo fuente
source_nb_path = os.path.join(DIRS['source'], ASSIGNMENT_ID, f"{ASSIGNMENT_ID}.ipynb")

if not os.path.exists(source_nb_path):
    print(f"❌ ERROR: No se encuentra el notebook fuente")
    print(f"\n📁 Esperado en: {source_nb_path}")
    print(f"\n💡 Debes crear primero el assignment maestro y subirlo a:")
    print(f"   {os.path.dirname(source_nb_path)}/")
else:
    print(f"✅ Archivo fuente encontrado: {source_nb_path}")
    print(f"\n🚀 Generando assignment para estudiantes...\n")
    print("="*70)
    
    # Generar assignment
    !nbgrader generate_assignment --assignment_id='{ASSIGNMENT_ID}' --debug
    
    print("="*70)
    release_path = os.path.join(DIRS['release'], ASSIGNMENT_ID, f"{ASSIGNMENT_ID}.ipynb")
    print(f"\n✅ Assignment generado en: {release_path}")
    print(f"\n💡 Comparte este archivo con tus estudiantes")

## 9. Autograding Masivo

**Prerequisito:** Los estudiantes deben haber subido sus notebooks en:
```
submitted/{estudiante}/{ASSIGNMENT_ID}/{ASSIGNMENT_ID}.ipynb
```

In [ ]:
# Verificar que hay submissions
print("🔍 Verificando submissions...\n")

submissions_encontradas = []
submissions_faltantes = []

for estudiante in estudiantes:
    nb_path = os.path.join(DIRS['submitted'], estudiante, ASSIGNMENT_ID, f"{ASSIGNMENT_ID}.ipynb")
    if os.path.exists(nb_path):
        submissions_encontradas.append(estudiante)
        print(f"✓ {estudiante}")
    else:
        submissions_faltantes.append(estudiante)
        print(f"✗ {estudiante} - NO ENCONTRADO")

print(f"\n📊 Resumen:")
print(f"   ✓ Submissions encontradas: {len(submissions_encontradas)}/{len(estudiantes)}")
print(f"   ✗ Submissions faltantes:   {len(submissions_faltantes)}/{len(estudiantes)}")

if submissions_faltantes:
    print(f"\n⚠️  Estudiantes sin submission:")
    for est in submissions_faltantes:
        print(f"   - {est}")

if not submissions_encontradas:
    print(f"\n❌ No hay submissions para calificar")
else:
    print(f"\n✅ Listo para calificar {len(submissions_encontradas)} estudiantes")

In [ ]:
# Autograding para todos los estudiantes con submissions
import subprocess

if not submissions_encontradas:
    print("❌ No hay submissions para calificar")
else:
    errores = []
    exitosos = []

    print(f"\n🚀 Iniciando calificación de {len(submissions_encontradas)} estudiantes...\n")
    print("="*70)

    for i, estudiante in enumerate(submissions_encontradas, 1):
        print(f"\n[{i}/{len(submissions_encontradas)}] 📝 Calificando: {estudiante}")
        print("-"*70)
        
        try:
            cmd = f"nbgrader autograde --student {estudiante} {ASSIGNMENT_ID}"
            result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
            
            if result.returncode == 0:
                exitosos.append(estudiante)
                print(f"✅ {estudiante} - OK")
            else:
                errores.append(estudiante)
                print(f"❌ {estudiante} - ERROR")
                if result.stderr:
                    print(f"   Error: {result.stderr[:200]}")
        except Exception as e:
            errores.append(estudiante)
            print(f"❌ {estudiante} - EXCEPTION: {e}")

    print("\n" + "="*70)
    print("📊 RESUMEN DE CALIFICACIÓN")
    print("="*70)
    print(f"✅ Exitosos: {len(exitosos)}/{len(submissions_encontradas)}")
    print(f"❌ Errores:  {len(errores)}/{len(submissions_encontradas)}")

    if exitosos:
        print(f"\n✅ Estudiantes calificados exitosamente:")
        for est in exitosos:
            print(f"   ✓ {est}")
    
    if errores:
        print(f"\n❌ Estudiantes con errores:")
        for est in errores:
            print(f"   ✗ {est}")
    
    print("="*70)

## 10. Generar Feedback Masivo

In [ ]:
# Generar feedback para todos los estudiantes
print(f"🚀 Generando feedback para: {ASSIGNMENT_ID}\n")
print("="*70)

!nbgrader feedback --assignment '{ASSIGNMENT_ID}'

print("="*70)
print(f"\n✅ Feedback generado en: {DIRS['feedback']}")
print(f"\n💡 Los estudiantes pueden ver su feedback en formato HTML")

## 11. Exportar Notas a CSV

In [ ]:
# Exportar calificaciones
OUTPUT_CSV = f"notas_{COURSE_ID}_{ASSIGNMENT_ID}.csv"

print(f"📊 Exportando calificaciones a: {OUTPUT_CSV}\n")
!nbgrader export --to '{OUTPUT_CSV}'

# Verificar que se creó el archivo
if os.path.exists(OUTPUT_CSV):
    print(f"\n✅ Archivo creado: {OUTPUT_CSV}")
    
    # Mostrar preview
    df_notas = pd.read_csv(OUTPUT_CSV)
    print(f"\n📋 Preview de calificaciones ({len(df_notas)} filas):")
    print("="*70)
    print(df_notas.head(10))
    print("="*70)
    
    # Descargar archivo
    print(f"\n📥 Descargando archivo...")
    from google.colab import files
    files.download(OUTPUT_CSV)
    print(f"✅ Descarga completada")
else:
    print(f"❌ Error: No se pudo crear el archivo de exportación")

## 12. Utilidades de Mantenimiento

### 12.1 Eliminar Submission de Estudiante Específico

In [ ]:
# Eliminar submission específica (para permitir re-submission)
print("⚠️  Esta acción eliminará la submission de un estudiante de la base de datos")
print("   (Permitirá que el estudiante vuelva a entregar)\n")

estudiante_eliminar = input(f"Nombre del estudiante (o ENTER para cancelar): ").strip()

if estudiante_eliminar:
    if estudiante_eliminar in estudiantes:
        !nbgrader db student remove '{estudiante_eliminar}' --assignment '{ASSIGNMENT_ID}' --force
        print(f"\n✅ Submission eliminada para: {estudiante_eliminar}")
    else:
        print(f"\n❌ Error: '{estudiante_eliminar}' no está en la lista de estudiantes")
else:
    print("Cancelado")

### 12.2 Limpiar Directorios

In [ ]:
# Función para limpiar un directorio
def limpiar_directorio(dir_path, dir_name):
    """Elimina todo el contenido de un directorio pero no el directorio mismo"""
    if not os.path.exists(dir_path):
        print(f"⚠️  El directorio {dir_name} no existe")
        return
    
    items_eliminados = 0
    for item in os.listdir(dir_path):
        item_path = os.path.join(dir_path, item)
        try:
            if os.path.isfile(item_path) or os.path.islink(item_path):
                os.unlink(item_path)
            elif os.path.isdir(item_path):
                shutil.rmtree(item_path)
            items_eliminados += 1
        except Exception as e:
            print(f"   ✗ Error eliminando {item}: {e}")
    
    print(f"✅ {dir_name}: {items_eliminados} items eliminados")

# Limpiar directorios (CUIDADO: Esto elimina archivos)
print("⚠️  ADVERTENCIA: Esto eliminará el contenido de los directorios\n")
print("Opciones:")
print("  1. Limpiar feedback")
print("  2. Limpiar autograded")
print("  3. Limpiar release")
print("  4. Limpiar TODO (feedback + autograded + release)")
print("  0. Cancelar\n")

opcion = input("Selecciona una opción: ").strip()

if opcion == '1':
    limpiar_directorio(DIRS['feedback'], 'feedback')
elif opcion == '2':
    limpiar_directorio(DIRS['autograded'], 'autograded')
elif opcion == '3':
    limpiar_directorio(DIRS['release'], 'release')
elif opcion == '4':
    print("\nLimpiando todos los directorios...\n")
    limpiar_directorio(DIRS['feedback'], 'feedback')
    limpiar_directorio(DIRS['autograded'], 'autograded')
    limpiar_directorio(DIRS['release'], 'release')
    print("\n✅ Limpieza completada")
else:
    print("Cancelado")

### 12.3 Ver Estadísticas del Assignment

In [ ]:
# Verificar estado de submissions
submitted_base = DIRS['submitted']

print(f"📊 Estado de submissions para: {ASSIGNMENT_ID}")
print("="*70)

submissions_ok = []
submissions_faltantes = []

for estudiante in estudiantes:
    assignment_path = os.path.join(submitted_base, estudiante, ASSIGNMENT_ID)
    
    if os.path.exists(assignment_path):
        files_in_dir = os.listdir(assignment_path)
        nb_files = [f for f in files_in_dir if f.endswith('.ipynb')]
        
        if nb_files:
            submissions_ok.append(estudiante)
            print(f"✓ {estudiante:40} → {len(nb_files)} archivo(s)")
        else:
            submissions_faltantes.append(estudiante)
            print(f"⚠  {estudiante:40} → carpeta vacía")
    else:
        submissions_faltantes.append(estudiante)
        print(f"✗ {estudiante:40} → no existe carpeta")

print("\n" + "="*70)
print("📊 RESUMEN")
print("="*70)
print(f"Total estudiantes:    {len(estudiantes)}")
print(f"✓ Con submissions:    {len(submissions_ok)} ({len(submissions_ok)/len(estudiantes)*100:.1f}%)")
print(f"✗ Sin submissions:    {len(submissions_faltantes)} ({len(submissions_faltantes)/len(estudiantes)*100:.1f}%)")

if submissions_faltantes:
    print(f"\n⚠️  Estudiantes sin submission:")
    for est in submissions_faltantes:
        print(f"   - {est}")

print("="*70)

## 13. Resumen de Configuración

Vista rápida de todos los parámetros configurados en esta sesión.

In [ ]:
# Resumen de configuración
print("="*70)
print("📋 RESUMEN DE CONFIGURACIÓN")
print("="*70)
print(f"\n🎓 Curso:")
print(f"   ID:              {COURSE_ID}")
print(f"   Assignment:      {ASSIGNMENT_ID}")
print(f"   Ruta base:       {BASE_PATH}")
print(f"   Timeout:         {TIMEOUT}s")

print(f"\n👥 Estudiantes:")
print(f"   Total:           {len(estudiantes)}")
print(f"   Con submissions: {len(submissions_ok) if 'submissions_ok' in locals() else 'N/A'}")

print(f"\n📁 Directorios:")
for name, path in DIRS.items():
    exists = "✓" if os.path.exists(path) else "✗"
    print(f"   {exists} {name:12} → {path}")

print(f"\n📄 Archivos:")
print(f"   Config:          /content/nbgrader_config.py")
print(f"   CSV estudiantes: {csv_filename if 'csv_filename' in locals() else 'N/A'}")
print(f"   Notas export:    {OUTPUT_CSV if 'OUTPUT_CSV' in locals() else 'N/A'}")

print("\n" + "="*70)
print("✅ Sistema configurado y listo")
print("="*70)